# Atelier Préparation de Données Images

**Contexte.** Une entreprise souhaite entraîner un modèle de Machine Learning / Deep Learning capable de reconnaître automatiquement le type de déchet présent sur une photographie (`cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`) afin d'améliorer le tri des déchets. Les images collectées proviennent de plusieurs sources et ne sont donc pas homogènes (dimensions, formats, modes couleur, qualité). L'objectif de cet atelier est de construire, à partir du dossier `data/raw/`, un jeu de données images propre et homogène (`data/cleaned/`), prêt à être utilisé pour l'entraînement.

## Imports et configuration

In [1]:
import os
import shutil
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError

RAW_DIR = Path("../data/raw")
CLEANED_DIR = Path("../data/cleaned")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(exist_ok=True)

CLASSES = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
CLASSES

['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

## Partie 1 – Exploration du dataset

### Fonction d'extraction des métadonnées d'une image

In [2]:
def extraire_metadonnees(chemin: Path, classe: str) -> dict:
    """Extrait nom, classe, format, mode, largeur, hauteur, ecart-type des pixels,
    nombre de canaux et taille (octets) d'une image. Robuste aux fichiers corrompus :
    les champs dependant du contenu de l'image restent a None si la lecture echoue."""
    infos = {
        'nom': chemin.name,
        'classe': classe,
        'chemin': str(chemin),
        'taille_octets': chemin.stat().st_size,
        'format': None,
        'mode': None,
        'largeur': None,
        'hauteur': None,
        'ecart_type_pixels': None,
        'canaux': None,
    }
    try:
        with Image.open(chemin) as img:
            img.load()  # force le decodage complet : declenche une erreur si le fichier est corrompu
            infos['format'] = img.format
            infos['mode'] = img.mode
            infos['largeur'], infos['hauteur'] = img.size
            infos['canaux'] = len(img.getbands())
            infos['ecart_type_pixels'] = float(np.array(img).std())
    except Exception:
        pass
    return infos

### Construction du DataFrame d'audit sur l'ensemble du dataset (`data/raw/`)

In [3]:
lignes = []
for classe in CLASSES:
    for chemin in sorted((RAW_DIR / classe).iterdir()):
        if chemin.is_file():
            lignes.append(extraire_metadonnees(chemin, classe))

df_audit = pd.DataFrame(lignes)
df_audit['resolution'] = list(zip(df_audit['largeur'], df_audit['hauteur']))
print(f"Nombre total d'images dans data/raw/ : {len(df_audit)}")
df_audit.shape

Nombre total d'images dans data/raw/ : 1032


(1032, 11)

### Aperçu du DataFrame d'audit

In [4]:
df_audit.head()

,nom,classe,chemin,taille_octets,format,mode,largeur,hauteur,ecart_type_pixels,canaux,resolution
0,cardboard1.jpg,cardboard,..\data\raw\cardboard\cardboard1.jpg,17333,JPEG,RGB,512.0,384.0,40.588529,3.0,"(512.0, 384.0)"
1,cardboard10.jpg,cardboard,..\data\raw\cardboard\cardboard10.jpg,21683,JPEG,RGB,512.0,384.0,42.571288,3.0,"(512.0, 384.0)"
2,cardboard100.jpg,cardboard,..\data\raw\cardboard\cardboard100.jpg,14884,JPEG,RGB,512.0,384.0,46.108305,3.0,"(512.0, 384.0)"
3,cardboard101.jpg,cardboard,..\data\raw\cardboard\cardboard101.jpg,14289,JPEG,RGB,512.0,384.0,72.263996,3.0,"(512.0, 384.0)"
4,cardboard102.jpg,cardboard,..\data\raw\cardboard\cardboard102.jpg,18015,JPEG,RGB,512.0,384.0,48.388937,3.0,"(512.0, 384.0)"


In [5]:
df_audit.dtypes

nom                      str
classe                   str
chemin                   str
taille_octets          int64
format                   str
mode                     str
largeur              float64
hauteur              float64
ecart_type_pixels    float64
canaux               float64
resolution            object
dtype: object

In [6]:
df_audit.describe(include='all')

,nom,classe,chemin,taille_octets,format,mode,largeur,hauteur,ecart_type_pixels,canaux,resolution
count,1032,1032,1032,1032.000000,1026,1026,1026.000000,1026.000000,1026.000000,1026.000000,1032
unique,1029,6,1032,NaN,3,3,NaN,NaN,NaN,NaN,5
top,image-blanche-512x384.jpg,paper,..\data\raw\cardboard\cardboard1.jpg,NaN,JPEG,RGB,NaN,NaN,NaN,NaN,"(512.0, 384.0)"
freq,2,252,1,NaN,1006,1006,NaN,NaN,NaN,NaN,1013
mean,NaN,NaN,NaN,18967.135659,NaN,NaN,506.011696,379.571150,50.251721,3.013645,NaN
std,NaN,NaN,NaN,17316.322788,NaN,NaN,52.892232,39.116528,15.982024,0.158680,NaN
min,NaN,NaN,NaN,924.000000,NaN,NaN,32.000000,32.000000,1.572536,1.000000,NaN
25%,NaN,NaN,NaN,11729.250000,NaN,NaN,512.000000,384.000000,38.694855,3.000000,NaN
50%,NaN,NaN,NaN,15488.000000,NaN,NaN,512.000000,384.000000,49.877604,3.000000,NaN
75%,NaN,NaN,NaN,20835.500000,NaN,NaN,512.000000,384.000000,61.796555,3.000000,NaN


### Répartition des formats et modes rencontrés

In [7]:
df_audit['format'].value_counts(dropna=False)

format
JPEG    1006
PNG       18
NaN        6
GIF        2
Name: count, dtype: int64

In [8]:
df_audit['mode'].value_counts(dropna=False)

mode
RGB     1006
RGBA      18
NaN        6
P          2
Name: count, dtype: int64

**Réponse.** Le dossier `data/raw/` contient **1032 images** réparties sur **6 classes** (cardboard, glass, metal, paper, plastic, trash). Les formats rencontrés sont **JPEG** (1006 images), **PNG** (18 images) et **GIF** (2 images) ; **6 fichiers** ne peuvent pas être décodés (`format` = `NaN`) : ce sont les images corrompues qui seront confirmées en Partie 2. Côté modes couleur, on trouve **1006 images en RGB**, **18 en RGBA** (canal alpha) et **2 en mode `P`** (palette indexée, typiques des GIF). Le dataset n'est donc pas homogène ni en format, ni en mode couleur, ni en dimensions (voir Partie 4).